In [ ]:
import pandas as pd
from scipy.stats import pearsonr

data = pd.read_csv('$PATH/WM_ICV_age/dataset/freesurfer_output_origin.csv')

features = ['ICV', 'WM', 'GM', 'ratio']
target = 'doctor_predicted_age'

for feature in features:
    corr, p_value = pearsonr(data[feature], data[target])
    print(f"{feature} vs {target} : Pearson r = {corr:.4f}, p-value = {p_value:.4g}")
print('\n')
data = pd.read_csv('$PATH/WM_ICV_age/dataset/fsl_output_origin.csv')
for feature in features:
    corr, p_value = pearsonr(data[feature], data[target])
    print(f"{feature} vs {target} : Pearson r = {corr:.4f}, p-value = {p_value:.4g}")
print('\n')
data = pd.read_csv('$PATH/WM_ICV_age/dataset/sienax_output_origin.csv')
for feature in features:
    corr, p_value = pearsonr(data[feature], data[target])
    print(f"{feature} vs {target} : Pearson r = {corr:.4f}, p-value = {p_value:.4g}")    
print('\n')

In [ ]:
import pandas as pd
import numpy as np
from scipy.stats import pearsonr

def stratified_sample(df, target_col, n_bins=5, samples_per_bin=5):
    df['age_bin'] = pd.qcut(df[target_col], q=n_bins, labels=False, duplicates='drop')
    
    sampled_df = (
        df.groupby('age_bin', group_keys=False)
        .apply(lambda x: x.sample(min(len(x), samples_per_bin), random_state=42),)
    )
    
    return sampled_df.drop(columns='age_bin')

def stratified_sampling_by_age_range(df, age_col='doctor_predicted_age', samples_per_bin=10):
    bins = [(0, 5), (6, 10), (11, 15), (16, 20), (21, 24)]
    
    sampled_df_list = []
    
    for i, (start, end) in enumerate(bins):
        bin_df = df[(df[age_col] >= start) & (df[age_col] <= end)]
        sampled = bin_df.sample(n=min(len(bin_df), samples_per_bin), random_state=42)
        sampled_df_list.append(sampled)

    final_df = pd.concat(sampled_df_list).reset_index(drop=True)
    return final_df

def compute_correlations(path, label):
    df = pd.read_csv(path)
    df = stratified_sampling_by_age_range(df, age_col='doctor_predicted_age', samples_per_bin=40)
    
    print(f"--- {label} ---")
    for feature in ['ICV', 'WM', 'GM', 'ratio']:
        corr, p_value = pearsonr(df[feature], df['doctor_predicted_age'])
        print(f"{feature} vs doctor_predicted_age : Pearson r = {corr:.4f}, p = {p_value:.4g}")
    print()

compute_correlations('$PATH/WM_ICV_age/dataset/freesurfer_output_origin.csv', label="FreeSurfer")
compute_correlations('$PATH/WM_ICV_age/dataset/fsl_output_origin.csv', label="FSL")
compute_correlations('$PATH/WM_ICV_age/dataset/sienax_output_origin.csv', label="SIENAX")


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import pearsonr

# File paths for the CSVs
file_paths = {
    "freesurfer": "$PATH/WM_ICV_age/dataset/freesurfer_output_origin.csv",
}

# Features and target column
features = ['WM', 'ICV']
target = 'doctor_predicted_age'

# Set seaborn style
sns.set(style="whitegrid")
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Plot each feature in a single subplot with data from all sources
for i, feature in enumerate(features):
    ax = axes[i]
    for label, path in file_paths.items():
        data = pd.read_csv(path)
        sns.regplot(x=data[feature], y=data[target], ax=ax,
                    scatter_kws={'s': 40, 'alpha': 0.6}, label=label,
                    line_kws={'linewidth': 2, 'color':"red"})
        corr, p_val = pearsonr(data[feature], data[target])
        ax.text(0.05, 0.9 - 0.1 * list(file_paths.keys()).index(label), 
                f"r={corr:.2f}, p={p_val:.2g}", 
                transform=ax.transAxes, fontsize=12)
    
    ax.set_title(f"{feature} vs. Biological Age")
    ax.set_xlabel(feature)
    ax.set_ylabel("biological ages (month)")

plt.tight_layout(rect=[0, 0.03, 1, 0.95])
plt.show()

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import pearsonr
from matplotlib.ticker import FuncFormatter

# File paths for the CSVs
file_paths = {
    "freesurfer": "$PATH/WM_ICV_age/dataset/freesurfer_output_origin.csv",
}

# Features and target column
features = ['WM', 'ICV']
target = 'doctor_predicted_age'

# Function to format x-axis ticks in thousands (e.g., 50000 → 50k)
def thousands_formatter(x, pos):
    return f'{int(x/1000)}k'

# Set seaborn style
sns.set(style="whitegrid")
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Plot each feature in a single subplot
for i, feature in enumerate(features):
    ax = axes[i]
    for label, path in file_paths.items():
        data = pd.read_csv(path)
        sns.regplot(x=data[feature], y=data[target], ax=ax,
                    scatter_kws={'s': 40, 'alpha': 0.6}, label=label,
                    line_kws={'linewidth': 2, 'color': "red"})
        corr, p_val = pearsonr(data[feature], data[target])
        ax.text(0.05, 0.9 - 0.1 * list(file_paths.keys()).index(label), 
                f"r={corr:.2f}, p={p_val:.2g}", 
                transform=ax.transAxes, fontsize=12)

    ax.set_title(f"{feature} vs. Biological Age")
    ax.set_xlabel(feature)
    ax.set_ylabel("biological ages (month)")
    ax.xaxis.set_major_formatter(FuncFormatter(thousands_formatter))

plt.tight_layout()
plt.show()


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import pearsonr
from matplotlib.ticker import FuncFormatter

file_paths = {
    "freesurfer": "$PATH/WM_ICV_age/dataset/freesurfer_output_origin.csv",
}

features = ['WM', 'ICV']
target = 'doctor_predicted_age'

def smart_formatter(x, pos):
    if x >= 1_000_000:
        return f'{x/1_000_000:.1f}M'
    elif x >= 1_000:
        return f'{x/1_000:.0f}k'
    else:
        return str(int(x))

sns.set(style="whitegrid")
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

for i, feature in enumerate(features):
    ax = axes[i]
    for label, path in file_paths.items():
        data = pd.read_csv(path)
        sns.regplot(x=data[feature], y=data[target], ax=ax,
                    scatter_kws={'s': 40, 'alpha': 0.6}, label=label,
                    line_kws={'linewidth': 2, 'color': "red"})
        corr, p_val = pearsonr(data[feature], data[target])
        ax.text(0.05, 0.9 - 0.1 * list(file_paths.keys()).index(label), 
                f"r={corr:.2f}, p={p_val:.2g}", 
                transform=ax.transAxes, fontsize=12)

    ax.set_title(f"{feature} vs. Biological Age")
    ax.set_xlabel(feature)
    ax.set_ylabel("biological ages (month)")
    ax.xaxis.set_major_formatter(FuncFormatter(smart_formatter))

plt.tight_layout()
plt.show()


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import pearsonr
from matplotlib.ticker import FuncFormatter
import numpy as np

file_paths = {
    "freesurfer": "$PATH/WM_ICV/ESNR/freesurfer_output_origin.csv",
}

features = ['WMV', 'ICV']
target = 'doctor_predicted_age'

def scaled_formatter(scale):
    def formatter(x, pos):
        return f'{x / scale:.1f}'
    return formatter

scales = {'WMV': 1e5, 'ICV': 1e6}
sns.set(style="whitegrid")
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

for i, feature in enumerate(features):
    ax = axes[i]
    for label, path in file_paths.items():
        data = pd.read_csv(path)
        sns.regplot(x=data[feature], y=data[target], ax=ax,
                    scatter_kws={'s': 40, 'alpha': 0.6}, label=label,
                    line_kws={'linewidth': 2, 'color': "red"})
        corr, p_val = pearsonr(data[feature], data[target])
        ax.text(0.05, 0.9 - 0.1 * list(file_paths.keys()).index(label), 
                f"r={corr:.2f}, p={p_val:.2g}", 
                transform=ax.transAxes, fontsize=12)

    scale = scales[feature]
    ax.xaxis.set_major_formatter(FuncFormatter(scaled_formatter(scale)))

    ax.set_title(f"{feature} vs. Biological Age")
    ax.set_xlabel(f"{feature} (×10⁵)" if feature == 'WMV' else f"{feature} (×10⁶)")
    ax.set_ylabel("Biological Ages (month)")

plt.tight_layout()
plt.show()


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import pearsonr

# File paths for the CSVs
file_paths = {
    "freesurfer": "$PATH/WM_ICV_age/dataset/freesurfer_output_origin.csv",
    "fsl": "$PATH/WM_ICV_age/dataset/fsl_output_origin.csv",
    "sienax": "$PATH/WM_ICV_age/dataset/sienax_output_origin.csv"
}

# Features and target column
features = ['WM', 'ICV']
target = 'doctor_predicted_age'

# Set seaborn style
sns.set(style="whitegrid")
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle("Pearson Correlation: Features vs. Doctor Predicted Age", fontsize=16)

# Plot each feature in a single subplot with data from all sources
for i, feature in enumerate(features):
    ax = axes[i]
    for label, path in file_paths.items():
        data = pd.read_csv(path)
        sns.regplot(x=data[feature], y=data[target], ax=ax,
                    scatter_kws={'s': 40, 'alpha': 0.6}, label=label,
                    line_kws={'linewidth': 2})
        corr, p_val = pearsonr(data[feature], data[target])
        ax.text(0.05, 0.9 - 0.1 * list(file_paths.keys()).index(label), 
                f"{label}: r={corr:.2f}, p={p_val:.2g}", 
                transform=ax.transAxes, fontsize=10)
    
    ax.set_title(f"{feature} vs. Biological Age")
    ax.set_xlabel(feature)
    ax.set_ylabel("biological ages (month)")
    ax.legend()

plt.tight_layout(rect=[0, 0.03, 1, 0.95])
plt.show()
